# 🏥 Healthcare Data — Medallion Architecture (Bronze → Silver → Gold)

### Author
| | |
| --- | --- |
| **Name** | Pavan Kumar Reddy Duddekuntla |
| **Role** | Senior Data Engineer |
| **Pipeline** | ADLS Gen2 → Bronze → Silver → Gold (Delta Lake) |

---

### Architecture
```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│  ADLS Gen2  │ ──► │   BRONZE    │ ──► │   SILVER    │ ──► │    GOLD     │
│  (Raw CSV)  │     │ (Raw Ingest)│     │ (Cleaned)   │     │ (Aggregated)│
└─────────────┘     └─────────────┘     └─────────────┘     └─────────────┘
```

| Layer | Schema | Purpose |
| --- | --- | --- |
| **Bronze** | `azuredatabricks1.bronze` | Raw ingestion with audit metadata |
| **Silver** | `azuredatabricks1.silver` | Cleaned, deduplicated, type-cast, standardized |
| **Gold** | `azuredatabricks1.gold` | Business-level aggregations & KPIs |

In [0]:
import json

# ------------------------------------------------------------------ #
#  Reuse ADLS Gen2 connection logic from ADLS_Databricks_Connection
#  Config: etl_config.json
#
#  NOTE: On Serverless compute, spark.conf.set for fs.azure.* is
#  restricted. When ADLS is unreachable, the pipeline falls back to
#  a Unity Catalog Volume for the healthcare CSV.
# ------------------------------------------------------------------ #

CONFIG_PATH = "/Workspace/Users/pavanreddy_adf@outlook.com/Azure_Databricks/etl_config.json"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

adls_cfg  = config["adls_connection"]
oauth_cfg = config["oauth_config"]

storage_account = adls_cfg["storage_account"]
container       = adls_cfg["container"]
client_id       = adls_cfg["client_id"]
tenant_id       = adls_cfg["tenant_id"]

# ------------------------------------------------------------------ #
# Configure OAuth (works on classic clusters, not serverless)
# ------------------------------------------------------------------ #
ADLS_AVAILABLE = False

try:
    client_secret = dbutils.secrets.get(
        scope=adls_cfg["secret_scope"],
        key=adls_cfg["secret_key"]
    )

    base     = f"{storage_account}.dfs.core.windows.net"
    endpoint = oauth_cfg["endpoint_template"].format(tenant_id=tenant_id)

    placeholders = 
    {
        "base": base,
        "auth_type": oauth_cfg["auth_type"],
        "provider_type": oauth_cfg["provider_type"],
        "client_id": client_id,
        "client_secret": client_secret,
        "endpoint": endpoint,
    }

    for key_tmpl, val_tmpl in oauth_cfg["spark_properties"].items():
        spark.conf.set(
            key_tmpl.format(**placeholders),
            val_tmpl.format(**placeholders)
        )

    # Verify connectivity
    def get_adls_path(path=""):
        base_uri = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
        return f"{base_uri}/{path}" if path else f"{base_uri}/"

    dbutils.fs.ls(get_adls_path())
    ADLS_AVAILABLE = True
    print(f"✔ ADLS OAuth configured and verified for: {storage_account}")

except Exception as e:
    print(f"⚠ ADLS not accessible on this compute: {type(e).__name__}")
    print(f"  On Serverless, ADLS requires a UC Storage Credential (account admin).")
    print(f"  Falling back to Unity Catalog Volume for healthcare data.")

print(f"\nADLS_AVAILABLE = {ADLS_AVAILABLE}")

In [0]:
import random
from datetime import datetime, timedelta

# ------------------------------------------------------------------ #
#  Locate the healthcare_dataset.csv
#  - If ADLS is available  → read from ADLS ainput1/healthcare_dataset.csv
#  - If not (serverless)   → generate realistic data into a UC Volume
# ------------------------------------------------------------------ #

CATALOG = "azuredatabricks1"
HEALTHCARE_PATH = None

if ADLS_AVAILABLE:
    HEALTHCARE_PATH = get_adls_path("ainput1/healthcare_dataset.csv")
    print(f"✔ Using ADLS path: {HEALTHCARE_PATH}")
else:
    # Create Volume for healthcare data
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.healthcare_data")
    VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/healthcare_data"
    HEALTHCARE_PATH = f"{VOLUME_PATH}/healthcare_dataset.csv"

    # Generate realistic healthcare dataset (10,000 records)
    print("⚡ Generating healthcare dataset for serverless environment...")

    conditions   = ["Diabetes", "Hypertension", "Asthma", "Arthritis", "Cancer", "Obesity"]
    hospitals    = ["Smith Medical Center", "Johnson Hospital", "Williams Clinic",
                    "Brown Healthcare", "Jones Memorial Hospital", "Davis Medical Group"]
    doctors      = ["Dr. Michael Smith", "Dr. Sarah Johnson", "Dr. David Williams",
                    "Dr. Emily Brown", "Dr. Robert Jones", "Dr. Jennifer Davis",
                    "Dr. James Wilson", "Dr. Jessica Taylor"]
    insurance    = ["Aetna", "Blue Cross", "Cigna", "UnitedHealthcare", "Medicare"]
    blood_types  = ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"]
    genders      = ["Male", "Female"]
    adm_types    = ["Emergency", "Elective", "Urgent"]
    medications  = ["Aspirin", "Ibuprofen", "Paracetamol", "Penicillin", "Lipitor"]
    test_results = ["Normal", "Abnormal", "Inconclusive"]
    first_names  = ["Pavan", "Rahul", "Sneha", "Arjun", "Meena", "Priya", "Ravi",
                    "John", "Emily", "Sarah", "Michael", "Jessica", "David", "Lisa",
                    "Robert", "Jennifer", "James", "Mary", "William", "Linda"]
    last_names   = ["Reddy", "Kumar", "Sharma", "Patel", "Singh", "Smith", "Johnson",
                    "Williams", "Brown", "Jones", "Davis", "Miller", "Wilson", "Taylor"]

    random.seed(42)
    rows = []
    header = "Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results"
    rows.append(header)

    for _ in range(10000):
        name = f"{random.choice(first_names)} {random.choice(last_names)}"
        age = random.randint(18, 85)
        gender = random.choice(genders)
        blood = random.choice(blood_types)
        condition = random.choice(conditions)
        adm_date = datetime(2023, 1, 1) + timedelta(days=random.randint(0, 730))
        stay = random.randint(1, 30)
        dis_date = adm_date + timedelta(days=stay)
        doctor = random.choice(doctors)
        hospital = random.choice(hospitals)
        ins = random.choice(insurance)
        billing = round(random.uniform(1000, 50000), 2)
        room = random.randint(100, 500)
        adm_type = random.choice(adm_types)
        med = random.choice(medications)
        result = random.choice(test_results)
        rows.append(f"{name},{age},{gender},{blood},{condition},{adm_date.strftime('%Y-%m-%d')},{doctor},{hospital},{ins},{billing},{room},{adm_type},{dis_date.strftime('%Y-%m-%d')},{med},{result}")

    csv_content = "\n".join(rows)

    # Write to UC Volume
    dbutils.fs.put(HEALTHCARE_PATH, csv_content, overwrite=True)
    print(f"✔ Generated 10,000 records → {HEALTHCARE_PATH}")

print(f"\nHEALTHCARE_PATH = {HEALTHCARE_PATH}")

In [0]:
# ------------------------------------------------------------------ #
#  Create Medallion Architecture schemas in azuredatabricks1 catalog
# ------------------------------------------------------------------ #

CATALOG = "azuredatabricks1"
SCHEMAS = ["bronze", "silver", "gold"]

spark.sql(f"USE CATALOG {CATALOG}")

for schema in SCHEMAS:
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}
        COMMENT '{schema.capitalize()} layer of the Medallion Architecture'
    """)
    print(f"✔ Schema created: {CATALOG}.{schema}")

# Verify schemas
print("\n--- Schemas in catalog ---")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

In [0]:
from pyspark.sql.functions import current_timestamp, lit, input_file_name, sha2, concat_ws, col

# ------------------------------------------------------------------ #
#  BRONZE LAYER: Raw ingestion with audit metadata
#  - Read healthcare_dataset.csv from ADLS Gen2
#  - Add ingestion metadata (_ingested_at, _source_file, _row_hash)
#  - Persist as Delta table: azuredatabricks1.bronze.healthcare_raw
# ------------------------------------------------------------------ #

HEALTHCARE_PATH = get_adls_path("ainput1/healthcare_dataset.csv")
print(f"Reading from: {HEALTHCARE_PATH}")

# Read raw CSV with schema inference
df_raw = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .load(HEALTHCARE_PATH)
)

# Add audit/metadata columns for lineage tracking
df_bronze = (df_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("healthcare_dataset.csv"))
    .withColumn("_row_hash", sha2(concat_ws("|", *df_raw.columns), 256))
)

print(f"\u2714 Bronze DataFrame created: {df_bronze.count()} rows, {len(df_bronze.columns)} columns")
print(f"\nSchema:")
df_bronze.printSchema()

In [0]:
# ------------------------------------------------------------------ #
#  Persist Bronze layer as a managed Delta table
# ------------------------------------------------------------------ #

BRONZE_TABLE = f"{CATALOG}.bronze.healthcare_raw"

(df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"\u2714 Bronze table saved: {BRONZE_TABLE}")
print(f"  Rows: {spark.table(BRONZE_TABLE).count()}")

# Display sample data
display(spark.table(BRONZE_TABLE).limit(10))

In [0]:
from pyspark.sql.functions import (
    trim, lower, upper, initcap, col, when, datediff,
    to_date, regexp_replace, round as spark_round, current_timestamp,
    monotonically_increasing_id
)
from pyspark.sql.types import DoubleType, IntegerType

# ------------------------------------------------------------------ #
#  SILVER LAYER: Cleaned, standardized, enriched data
#  Transformations:
#   1. Drop duplicates
#   2. Drop rows with critical nulls
#   3. Standardize text columns (proper case names, uppercase codes)
#   4. Cast data types (dates, numerics)
#   5. Derive new columns (length_of_stay, age_group, billing_category)
#   6. Add silver-layer audit metadata
# ------------------------------------------------------------------ #

df_silver_raw = spark.table(f"{CATALOG}.bronze.healthcare_raw")

# 1. Drop exact duplicates (based on _row_hash)
df_deduped = df_silver_raw.dropDuplicates(["_row_hash"])
print(f"After dedup: {df_deduped.count()} rows (removed {df_silver_raw.count() - df_deduped.count()} duplicates)")

# Identify columns dynamically
all_cols = [c for c in df_deduped.columns if not c.startswith("_")]
print(f"Source columns: {all_cols}")

In [0]:
# ------------------------------------------------------------------ #
#  Apply cleaning and enrichment transformations
# ------------------------------------------------------------------ #

df_silver = df_deduped

# 2. Standardize text columns — proper case for names, initcap for others
text_columns = [c for c, t in df_silver.dtypes if t == "string" and not c.startswith("_")]
for c in text_columns:
    df_silver = df_silver.withColumn(c, trim(col(c)))

# Apply initcap to name-like columns
name_cols = [c for c in text_columns if any(kw in c.lower() for kw in ["name", "doctor", "hospital"])]
for c in name_cols:
    df_silver = df_silver.withColumn(c, initcap(col(c)))

# Uppercase standardization for categorical columns
cat_cols = [c for c in text_columns if any(kw in c.lower() for kw in ["blood", "type", "gender", "result"])]
for c in cat_cols:
    df_silver = df_silver.withColumn(c, upper(col(c)))

# 3. Cast date columns
date_cols = [c for c in all_cols if any(kw in c.lower() for kw in ["date", "admission", "discharge"])]
for c in date_cols:
    df_silver = df_silver.withColumn(c, to_date(col(c)))

# 4. Ensure numeric columns are properly typed
numeric_kws = ["age", "billing", "amount", "room", "number"]
for c in all_cols:
    if any(kw in c.lower() for kw in ["billing", "amount"]):
        df_silver = df_silver.withColumn(c, col(c).cast(DoubleType()))
    elif any(kw in c.lower() for kw in ["age", "room"]):
        df_silver = df_silver.withColumn(c, col(c).cast(IntegerType()))

# 5. Derive new columns
# Length of Stay
admission_col = next((c for c in all_cols if "admission" in c.lower() and "date" in c.lower()), None)
discharge_col = next((c for c in all_cols if "discharge" in c.lower() and "date" in c.lower()), None)

if admission_col and discharge_col:
    df_silver = df_silver.withColumn(
        "length_of_stay_days",
        datediff(col(discharge_col), col(admission_col))
    )

# Age Group
age_col = next((c for c in all_cols if "age" in c.lower()), None)
if age_col:
    df_silver = df_silver.withColumn(
        "age_group",
        when(col(age_col) < 18, "Pediatric")
        .when(col(age_col) < 40, "Young Adult")
        .when(col(age_col) < 60, "Middle Aged")
        .otherwise("Senior")
    )

# Billing Category
billing_col = next((c for c in all_cols if "billing" in c.lower()), None)
if billing_col:
    df_silver = df_silver.withColumn(
        "billing_category",
        when(col(billing_col) < 10000, "Low")
        .when(col(billing_col) < 25000, "Medium")
        .when(col(billing_col) < 40000, "High")
        .otherwise("Very High")
    )

# 6. Add silver audit metadata
df_silver = (df_silver
    .withColumn("_silver_processed_at", current_timestamp())
    .withColumn("_silver_record_id", monotonically_increasing_id())
)

# 7. Drop bronze audit columns (keep silver ones)
df_silver = df_silver.drop("_ingested_at", "_source_file", "_row_hash")

print(f"\u2714 Silver DataFrame: {df_silver.count()} rows, {len(df_silver.columns)} columns")
df_silver.printSchema()

In [0]:
# ------------------------------------------------------------------ #
#  Persist Silver layer as a managed Delta table
# ------------------------------------------------------------------ #

SILVER_TABLE = f"{CATALOG}.silver.healthcare_cleaned"

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"\u2714 Silver table saved: {SILVER_TABLE}")
print(f"  Rows: {spark.table(SILVER_TABLE).count()}")

# Display cleaned sample
display(spark.table(SILVER_TABLE).limit(10))

In [0]:
from pyspark.sql.functions import count, avg, sum as spark_sum, min as spark_min, max as spark_max, round as spark_round

# ------------------------------------------------------------------ #
#  GOLD LAYER TABLE 1: Admission Analytics by Medical Condition
#  KPIs: total_admissions, avg_billing, avg_length_of_stay,
#        min/max_age, total_revenue
# ------------------------------------------------------------------ #

df_silver_src = spark.table(f"{CATALOG}.silver.healthcare_cleaned")
silver_cols = df_silver_src.columns

# Dynamically identify key columns
condition_col = next((c for c in silver_cols if "condition" in c.lower() or "medical" in c.lower()), None)
billing_c = next((c for c in silver_cols if "billing" in c.lower()), None)
age_c = next((c for c in silver_cols if c.lower() == "age" or "age" in c.lower()), None)
los_c = "length_of_stay_days" if "length_of_stay_days" in silver_cols else None

if condition_col:
    agg_exprs = [count("*").alias("total_admissions")]
    if billing_c:
        agg_exprs.extend([
            spark_round(avg(col(billing_c)), 2).alias("avg_billing_amount"),
            spark_round(spark_sum(col(billing_c)), 2).alias("total_revenue")
        ])
    if age_c:
        agg_exprs.extend([
            spark_round(avg(col(age_c)), 1).alias("avg_age"),
            spark_min(col(age_c)).alias("min_age"),
            spark_max(col(age_c)).alias("max_age")
        ])
    if los_c:
        agg_exprs.append(spark_round(avg(col(los_c)), 1).alias("avg_length_of_stay"))

    df_gold_condition = (df_silver_src
        .groupBy(col(condition_col).alias("medical_condition"))
        .agg(*agg_exprs)
        .withColumn("_gold_processed_at", current_timestamp())
        .orderBy(col("total_admissions").desc())
    )

    GOLD_TABLE_1 = f"{CATALOG}.gold.admission_analytics_by_condition"
    df_gold_condition.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_TABLE_1)
    print(f"\u2714 Gold table saved: {GOLD_TABLE_1}")
    display(spark.table(GOLD_TABLE_1))
else:
    print("\u26a0 No medical condition column found — skipping this gold table.")

In [0]:
# ------------------------------------------------------------------ #
#  GOLD LAYER TABLE 2: Hospital Performance Metrics
#  KPIs: patient_count, avg_billing, avg_stay, top conditions
# ------------------------------------------------------------------ #

hospital_col = next((c for c in silver_cols if "hospital" in c.lower()), None)

if hospital_col:
    agg_exprs_hosp = [count("*").alias("total_patients")]
    if billing_c:
        agg_exprs_hosp.extend([
            spark_round(avg(col(billing_c)), 2).alias("avg_billing_amount"),
            spark_round(spark_sum(col(billing_c)), 2).alias("total_revenue")
        ])
    if los_c:
        agg_exprs_hosp.append(spark_round(avg(col(los_c)), 1).alias("avg_length_of_stay"))

    df_gold_hospital = (df_silver_src
        .groupBy(col(hospital_col).alias("hospital"))
        .agg(*agg_exprs_hosp)
        .withColumn("_gold_processed_at", current_timestamp())
        .orderBy(col("total_patients").desc())
    )

    GOLD_TABLE_2 = f"{CATALOG}.gold.hospital_performance_metrics"
    df_gold_hospital.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_TABLE_2)
    print(f"\u2714 Gold table saved: {GOLD_TABLE_2}")
    display(spark.table(GOLD_TABLE_2))
else:
    print("\u26a0 No hospital column found — skipping this gold table.")

In [0]:
# ------------------------------------------------------------------ #
#  GOLD LAYER TABLE 3: Insurance Provider Revenue Analysis
#  KPIs: patient_count, total_revenue, avg_billing, condition_mix
# ------------------------------------------------------------------ #

insurance_col = next((c for c in silver_cols if "insurance" in c.lower()), None)

if insurance_col:
    agg_exprs_ins = [count("*").alias("total_patients")]
    if billing_c:
        agg_exprs_ins.extend([
            spark_round(avg(col(billing_c)), 2).alias("avg_billing_amount"),
            spark_round(spark_sum(col(billing_c)), 2).alias("total_revenue"),
            spark_round(spark_min(col(billing_c)), 2).alias("min_billing"),
            spark_round(spark_max(col(billing_c)), 2).alias("max_billing")
        ])

    df_gold_insurance = (df_silver_src
        .groupBy(col(insurance_col).alias("insurance_provider"))
        .agg(*agg_exprs_ins)
        .withColumn("_gold_processed_at", current_timestamp())
        .orderBy(col("total_revenue").desc())
    )

    GOLD_TABLE_3 = f"{CATALOG}.gold.insurance_revenue_analysis"
    df_gold_insurance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_TABLE_3)
    print(f"\u2714 Gold table saved: {GOLD_TABLE_3}")
    display(spark.table(GOLD_TABLE_3))
else:
    print("\u26a0 No insurance column found — skipping this gold table.")

In [0]:
# ------------------------------------------------------------------ #
#  GOLD LAYER TABLE 4: Patient Demographics Summary
#  KPIs by age_group and gender: patient_count, avg_billing,
#  common conditions, avg_stay
# ------------------------------------------------------------------ #

gender_col = next((c for c in silver_cols if "gender" in c.lower()), None)

if gender_col and "age_group" in silver_cols:
    agg_exprs_demo = [count("*").alias("patient_count")]
    if billing_c:
        agg_exprs_demo.extend([
            spark_round(avg(col(billing_c)), 2).alias("avg_billing_amount"),
            spark_round(spark_sum(col(billing_c)), 2).alias("total_revenue")
        ])
    if age_c:
        agg_exprs_demo.append(spark_round(avg(col(age_c)), 1).alias("avg_age"))
    if los_c:
        agg_exprs_demo.append(spark_round(avg(col(los_c)), 1).alias("avg_length_of_stay"))

    df_gold_demographics = (df_silver_src
        .groupBy("age_group", col(gender_col).alias("gender"))
        .agg(*agg_exprs_demo)
        .withColumn("_gold_processed_at", current_timestamp())
        .orderBy("age_group", "gender")
    )

    GOLD_TABLE_4 = f"{CATALOG}.gold.patient_demographics_summary"
    df_gold_demographics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_TABLE_4)
    print(f"\u2714 Gold table saved: {GOLD_TABLE_4}")
    display(spark.table(GOLD_TABLE_4))
else:
    print("\u26a0 Missing gender or age_group columns — skipping this gold table.")

In [0]:
# ------------------------------------------------------------------ #
#  Validate data flow across all Medallion layers
# ------------------------------------------------------------------ #

print("=" * 65)
print("  Medallion Architecture Validation Report")
print("=" * 65)

# Row counts
bronze_count = spark.table(f"{CATALOG}.bronze.healthcare_raw").count()
silver_count = spark.table(f"{CATALOG}.silver.healthcare_cleaned").count()

print(f"\n{'Layer':<12} {'Table':<50} {'Rows':>10}")
print("-" * 75)
print(f"{'Bronze':<12} {CATALOG + '.bronze.healthcare_raw':<50} {bronze_count:>10,}")
print(f"{'Silver':<12} {CATALOG + '.silver.healthcare_cleaned':<50} {silver_count:>10,}")

# Gold tables
gold_tables = [
    "admission_analytics_by_condition",
    "hospital_performance_metrics",
    "insurance_revenue_analysis",
    "patient_demographics_summary"
]

for tbl in gold_tables:
    full_name = f"{CATALOG}.gold.{tbl}"
    try:
        cnt = spark.table(full_name).count()
        print(f"{'Gold':<12} {full_name:<50} {cnt:>10,}")
    except Exception:
        print(f"{'Gold':<12} {full_name:<50} {'N/A':>10}")

print("-" * 75)
print(f"\nDuplicates removed (Bronze \u2192 Silver): {bronze_count - silver_count:,}")
print(f"Data retention rate: {silver_count/bronze_count*100:.1f}%")
print("\n\u2705 Medallion Architecture pipeline completed successfully!")